# Variational problems: Sample-based quantum diagonalization

*Usage estimate of QPU time for the notebook excluding the bonus part: 10 seconds on Nighthawk r1 or 2 seconds on Heron r2. (NOTE: This is an estimate only. Your runtime might vary.)*

*The QPU usage estimate reflects only backend execution time. Queueing, calibration, and runtime session delays can extend total execution to several minutes, even for workloads using only seconds of QPU time.*

### Table of Contents

- Exercise 1: Map LUCJ circuit to the Nighthawk device
- Exercise 2: Examine the resulting bitstring
- Exercise 3: Recover the bitstring
- Exercise 4: Augment SQD with a classical reference subspace
- Scored Exercise: Find your best subspace!

In Lab 4a, we explored what quantum advantage means and how the [Quantum Advantage Tracker](https://quantum-advantage-tracker.github.io/) classifies demonstrations into three categories: classically verifiable problems, variational problems, and observable estimation. We saw examples like the $ \mathrm{Fe}_4\mathrm{S}_4$ molecule computation (variational) and the Loschmidt Echo calculation (observable estimation).

Now we look into the second category: **variational problems**. These are chemistry and material science problems where we are focused on obtaining ground-state for advanced simulation. Materials are governed directly by quantum mechanics and this obviously makes them an ideal domain for applying quantum computataion.

In this lab, you will estimate the **ground-state energy** of the nitrogen molecule (N₂) on a real quantum computer using **Sample-based Quantum Diagonalization (SQD)**.

SQD is a hybrid quantum–classical method: the quantum computer runs a parameterized circuit and *proposes* which electron arrangements are most important, while the classical computer does the linear algebra — diagonalizing the molecular Hamiltonian in that small, smartly chosen set of configurations.

By the end of this chapter you will have mapped a chemistry circuit to two different IBM device topologies, cleaned up noisy hardware bitstrings, and beaten the classical brute-force method using real quantum hardware.

### Install the necessary libraries

In [ ]:
%pip install qiskit-addon-sqd
%pip install ffsim

### Imports

In [ ]:
# Import necessary packages
import warnings
warnings.filterwarnings("ignore")

import copy
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import cast
from itertools import combinations
from functools import partial

import pyscf
import pyscf.cc
import pyscf.mcscf
import ffsim

from qiskit.circuit import QuantumCircuit, QuantumRegister, CircuitInstruction, Barrier
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_error_map
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit_ibm_runtime.fake_provider import FakeMiami

from qiskit_addon_sqd.fermion import SCIResult, solve_sci_batch
from qiskit_addon_sqd.subsampling import subsample
from qiskit_addon_sqd.counts import bit_array_to_arrays, bitstring_matrix_to_integers

In [ ]:
from qc_grader.challenges.qgss_2026 import (
    check_progress,
    grade_lab4c_ex1a,
    grade_lab4c_ex1b,
    grade_lab4c_ex2a,
    grade_lab4c_ex2b,
    grade_lab4c_ex3a,
    grade_lab4c_ex3b,
    grade_lab4c_ex4,
    grade_lab4c_exbonus,
)

Throughout the Lab you can use the `check_progress` function to see how many exercises you have completed:

In [ ]:
check_progress()

## Background: Molecular quantum chemistry in 5 minutes

You do **not** need a chemistry background to complete this lab. This section gives you just enough vocabulary.

### Molecules, orbitals, and electrons

Think of a molecule as a small concert hall.
The **molecular orbitals** are the seats in that hall.
The **electrons** are the guests who must be seated according to one strict rule: each seat (orbital) can hold *at most one guest of a given spin* — this is the Pauli exclusion principle.

Each spatial orbital can host two electrons, one with **spin-α (↑)** and one with **spin-β (↓)**.
We always track the two spin flavours separately, which is exactly why the bitstrings you will measure later are split into an α half and a β half.

### The ground state and why it matters

The **ground state** is the lowest-total-energy seating arrangement.
It tells chemists whether a molecule is stable, how strong its bonds are, and whether it will react.
Computing the ground-state energy accurately is one of the most compelling near-term applications of quantum computers because the number of possible seatings grows *exponentially* — for $n$ orbitals and $N_e$ electrons there are $\binom{n}{N_e}$ arrangements per spin species, which quickly exceeds any classical computer's reach.
For N₂ in this lab, with 26 active orbitals and 5 electrons per spin, that is already $\binom{26}{5} = 65{,}780$ configurations per spin sector — and the full Hilbert space dimension is the product of both spin sectors: over four billion states.

Finding the ground state is formally an **eigenvalue problem**:

$$\hat{H} |\psi_0\rangle = E_0 |\psi_0\rangle$$

where $|\psi_0\rangle$ is the ground-state wavefunction and $E_0$ is the lowest energy eigenvalue.
Equivalently, the **variational principle** frames it as a minimization:

$$E_0 = \min_{\|\psi\|=1} \langle \psi | \hat{H} | \psi \rangle$$

Any trial state $|\psi\rangle$ you construct gives an **upper bound** on $E_0$: $\langle\psi|\hat{H}|\psi\rangle \geq E_0$.
This inequality underlies every variational quantum algorithm, including SQD.

In second quantization the Hamiltonian separates into one- and two-body terms:

$$\hat{H} = \sum_{pq} h_{pq}\,\hat{a}^\dagger_p \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs}\,\hat{a}^\dagger_p \hat{a}^\dagger_q \hat{a}_s \hat{a}_r + E_\text{nuc}$$

- $h_{pq}$: **one-electron integrals** — kinetic energy and electron–nucleus attraction (`hcore` in the code).
- $g_{pqrs}$: **two-electron repulsion integrals** — electron–electron Coulomb repulsion (`eri` in the code).
- $E_\text{nuc}$: constant nuclear repulsion energy.

SQD solves this problem by projecting $\hat{H}$ onto a small **subspace** — a carefully chosen set of Slater determinants — and diagonalizing it exactly within that subspace.
The quantum computer's job is to propose which determinants belong there.

### Classical reference methods

Three classical benchmarks frame our plots:

| Method | What it does | Accuracy |
|---|---|---|
| **Hartree–Fock (HF)** | Each electron ignores every other and picks the cheapest seat independently. Fast, but ignores electron *correlation* — the tendency of electrons to dodge each other. | Upper bound on $E_0$; rough estimation. |
| **CCSD** (Coupled Cluster Singles & Doubles) | Starts from HF, then adds 1- and 2-electron "jumps" out of the HF arrangement using algebraic amplitudes $t_1$ (singles) and $t_2$ (doubles). Captures most correlation. | Provides efficient and reasonable classical estimation of $E_0$. |
| **Classical selected-CI** (Reference subspace in this lab) | Builds and diagonalizes the Hamiltonian in a small, hand-picked set of Slater determinants. When high-quality determinants are provided, it can approach near-exact accuracy. The reference subspace in this lab is a brute-force proxy: configurations are enumerated by excitation rank with no selection criterion, then diagonalized exactly. If your SQD result beats this baseline, you have demonstrated *quantum utility*: the quantum sampler has found configurations that no brute-force enumeration could reach. | Scales with configuration quality. The lab's brute-force version is typically better than HF but short of CCSD. |

> **Key point:** We will also *use* CCSD to seed our quantum circuit — its amplitudes describe how electrons want to jump between orbitals, and we transplant that chemical intuition directly into the parameters of the quantum ansatz.

## 1. Building the N₂ molecule and its Hamiltonian

### The nitrogen molecule

We study **molecular nitrogen (N₂)**: two nitrogen atoms separated by 2.0 Å.
This is roughly double N₂'s equilibrium bond length (~1.1 Å) — a stretched geometry where electron correlation effects are more pronounced, making the gap between HF and the exact ground state larger and the quantum computing application more compelling.
We describe the molecule using the **`cc-pvdz` basis set** — a standard vocabulary of orbital shapes that balances accuracy with computational cost.
Running Hartree–Fock in this basis gives us 28 spatial orbitals in total.

### Active space and frozen orbitals

Not all orbitals matter equally.
The innermost **core** orbitals (the 1s electrons of each nitrogen atom) sit deep in energy and essentially never change — they are perfectly happy where they are no matter what the molecule does.
Computing those orbitals quantum-mechanically would waste precious qubits.

We therefore **freeze** them:
- `n_frozen = 2` removes the 2 lowest-energy core orbitals from the quantum treatment.
- `active_space = range(n_frozen, mol.nao_nr())` is the set of orbitals where the chemically interesting action — bond formation, electron correlation — actually happens.

This is the **active-space approximation**: solve the full HF over everything, then only treat the active-space electrons on the quantum computer.
After freezing, we are left with:
- `num_orbitals` active spatial orbitals → this will equal the number of qubits per spin sector.
- `(num_elec_a, num_elec_b)` active electrons of spin-α and spin-β.  N₂ is a closed-shell singlet, so `num_elec_a == num_elec_b`.

The code cell below also extracts the **one-electron integrals** (`hcore`), **two-electron repulsion integrals** (`eri`), and the constant **nuclear repulsion energy** — together these are the numerical representation of $\hat{H}$ restricted to the active space, which the classical eigensolver will diagonalize later.

In [ ]:
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (2.0, 0, 0)]],
    basis="cc-pvdz",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

# Print molecular information
print(f"Number of molecular orbitals: {num_orbitals}")
print(f"Number of electrons (a, b): {(num_elec_a, num_elec_b)}")

### Running CCSD to seed the quantum ansatz

Before we build the quantum circuit, we run **CCSD** classically.
We are *not* using CCSD as the final answer — that would defeat the purpose.
We are **mining it for its amplitudes**:

- `t1[i, a]` — the amplitude for a single electron to "jump" from occupied orbital $i$ to virtual orbital $a$.
- `t2[i, j, a, b]` — the amplitude for an electron *pair* to scatter from $(i, j)$ to $(a, b)$.

These amplitudes encode CCSD's chemical intuition about how electrons correlate.
In the next section, the `ffsim` library will read them and compile them into the rotation angles of the quantum circuit, effectively transplanting classical chemistry knowledge into the quantum device as a high-quality starting guess.

In [ ]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run(max_cycle=1000)
t1 = ccsd.t1
t2 = ccsd.t2

## 2. Quantum state preparation: the LUCJ ansatz

### What is the LUCJ ansatz?

The goal of state preparation is to build a quantum circuit that approximates the N₂ ground state well enough that sampling it yields electron configurations close to the true ground state.
We use the **LUCJ ansatz** — **Local Unitary Cluster Jastrow** — a hardware-efficient parameterized circuit that captures electron correlation through two ingredients:

1. **Orbital rotations** $e^{i\hat{K}}$: one-body unitaries that mix the molecular orbitals (think of them as rearranging the seats).
2. **Jastrow factor** $e^{i\sum_{pq} J_{pq}\,\hat{n}_p \hat{n}_q}$: a diagonal interaction that penalizes or rewards pairs of electrons simultaneously occupying orbitals $p$ and $q$ (electrons react to each other's presence).

One UCJ layer has the form:

$$U_\mu = e^{i\hat{K}_\mu}\; e^{i\sum_{p<q} J^{(\mu)}_{pq}\,\hat{n}_p \hat{n}_q}\; e^{-i\hat{K}_\mu}$$

where $\hat{n}_p = a^\dagger_p a_p$ is the occupation-number operator for orbital $p$.
Here $a^\dagger_p$ **creates** an electron in orbital $p$ and $a_p$ **removes** one — this is the standard **second-quantization** language for many-body quantum systems.
$\hat{n}_p$ is simply the projector onto the occupied state: it equals 1 if orbital $p$ is occupied and 0 if empty.
The full ansatz stacks `n_reps` such layers: $U = U_{n_\text{reps}} \cdots U_1$.
Here we use `n_reps = 1`.

### From CCSD amplitudes to circuit parameters

`ffsim.UCJOpSpinBalanced.from_t_amplitudes(t1, t2, n_reps, interaction_pairs)` reads the CCSD single- and double-excitation amplitudes and compiles them into the rotation angles $K_\mu$ and Jastrow couplings $J_{pq}^{(\mu)}$.
Intuitively: if CCSD says "orbitals $i$ and $a$ swap strongly", the corresponding orbital rotation in $e^{i\hat{K}}$ gets a large angle.

**Spin-balanced** means the α (spin-up) and β (spin-down) sectors share the same orbital-rotation parameters — appropriate for a closed-shell singlet like N₂ (same α and β occupations), and it halves the number of free parameters.

### Interaction pairs — the hardware constraint

The Jastrow factor requires two qubits to interact directly.
The `interaction_pairs` argument tells `ffsim` which pairs of orbitals can interact:

- `alpha_alpha_indices = [(p, p+1) ...]` — same-spin *nearest-neighbor* interactions along the qubit chain (one line per spin sector on the chip).
- `alpha_beta_indices` — **cross-spin** interactions: an α qubit at position $p$ couples to a β qubit at position $q$.
  These are **constrained by the device wiring**: the coupling is only hardware-native if the two physical qubits are neighbors on the chip.

This is the key point of Exercise 1: different IBM devices have different topologies, so the allowed `alpha_beta_indices` differ between a Heron chip and a Nighthawk chip.

In [ ]:
service = QiskitRuntimeService()
backend_hr = service.backend("ibm_kingston")
plot_error_map(backend_hr)

### Jordan–Wigner encoding: from orbitals to qubits

The **Jordan–Wigner (JW) transformation** is the dictionary that maps fermionic orbital states onto qubit states:

$$\text{orbital } p \text{ occupied} \;\Longleftrightarrow\; \text{qubit } p = |1\rangle, \qquad
  \text{orbital } p \text{ empty} \;\Longleftrightarrow\; \text{qubit } p = |0\rangle$$

We use $2 \times \texttt{num\_orbitals}$ qubits in total:
- Qubits $0, \ldots, \texttt{num\_orbitals}-1$ → spin-α occupations
- Qubits $\texttt{num\_orbitals}, \ldots, 2\cdot\texttt{num\_orbitals}-1$ → spin-β occupations

This is exactly why, much later in Exercise 2a, a measured bitstring of length `2*num_orbitals` splits cleanly into an α half and a β half.

Two `ffsim` circuit instructions handle the encoding:
- `PrepareHartreeFockJW` — sets the initial qubit state to the Hartree–Fock occupation pattern (the first `num_elec_a` α-qubits and `num_elec_b` β-qubits are $|1\rangle$, the rest $|0\rangle$).
- `UCJOpSpinBalancedJW` — applies the UCJ correlator in the JW qubit basis.

### PRE_INIT passes and gate counts

The `ffsim.qiskit.PRE_INIT` transpiler pass decomposes the high-level `ffsim` gate objects into native hardware gates *before* the main transpilation pass runs.
Running `count_ops()` with and without `PRE_INIT` shows how many native 2-qubit gates the circuit requires — fewer gates means less accumulated noise.

The code below shows the complete setup for the **Heron (`ibm_kingston`)** device as a worked example.
Exercise 1 asks you to redo this mapping for the **Nighthawk (`ibm_miami`)** device.

### The SQD workflow: a brief map

Once the circuit is ready, the rest of this chapter follows a four-step loop:

1. **Sample** — run the LUCJ circuit on hardware and collect bitstrings.
2. **Recover configurations** — fix bitstrings that noise has pushed to the wrong electron count.
3. **Build a subspace** — collect the most frequent (most probable) electron configurations as a basis.
4. **Diagonalize** $\hat{H}$ in that subspace — get an energy estimate; feed the resulting orbital occupancies back to step 2.

Each iteration the subspace gets a little smarter.

In [ ]:
# For the Heron device, the alpha-beta interaction happens per every four indicies
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices_hr = [(p, p) for p in range(0, num_orbitals, 4)]
alpha_beta_indices_hr = alpha_beta_indices_hr[:5]  # truncate at fifth pair

print(alpha_beta_indices_hr)

In [ ]:
n_reps = 1
ucj_op_hr = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_hr),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# create an empty quantum circuit
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_hr = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit_hr.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# apply the UCJ operator to the reference state
circuit_hr.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_hr), qubits)

# investigate into the circuit for execution
inner_circuit_hr = ffsim.qiskit.PRE_INIT.run(circuit_hr)
for i in range(2, 0, -1):
    inner_circuit_hr.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_hr.decompose().draw("mpl", fold=-1))

# insert final measurement
circuit_hr.measure_all()

In [ ]:
# select spin a and b layout according to the device topology
spin_a_layout = [
     21,  36,  41,  42,  43,     56,  63,  64,  65,  77,
     85,  86,  87,  97, 107,    108, 109, 118, 129, 128,
    127, 126, 125, 117, 105,    104
]
spin_b_layout = [
     23,  24,  25,  37,  45,     46,  47,  57,  67,  68,
     69,  78,  89,  90,  91,     98, 111, 112, 113, 114,
    115,  99,  95,  94,  93,     79
]

initial_layout = spin_a_layout + spin_b_layout

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_hr, initial_layout=initial_layout
)

# without PRE_INIT passes
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_hr.count_ops()}")

# with PRE_INIT passes
# We will use the circuit generated by this pass manager for hardware execution
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_hr.count_ops()}")

## Exercise 1: Map LUCJ circuit for the Nighthawk device

Now let's port the same circuit to the **Nighthawk (`ibm_miami`)** device.
Run the error-map cell below and compare the connectivity diagram to Heron's.
Notice how the α and β qubit rails are wired differently — that changes which cross-spin interaction pairs are hardware-native.

> **No access to `ibm_miami`?** Use `FakeMiami` instead — uncomment the second line in the cell below. `FakeMiami` reproduces the Nighthawk device topology (same qubit graph and basis gates) so the circuit mapping and layout exercises work identically; the only difference is that it simulates noise rather than running on real hardware.

In [ ]:
backend_nh = service.backend("ibm_miami")
# # Use this command to use FakeMiami if you don't have access to ibm_miami
# backend_nh = FakeMiami()

plot_error_map(backend_nh)

<a id="exercise_1a"></a>
<div class="alert alert-block alert-success">

<b>Exercise 1a: Select the alpha–beta interaction pairs for the Nighthawk device</b>

**Your Goal:** Fill `alpha_beta_indices_nh` with the cross-spin interaction pairs appropriate for the Nighthawk device topology.

**Background:** Recall from Section 2 that each `(p, q)` tuple in `alpha_beta_indices` means "the Jastrow factor couples α-orbital $p$ to β-orbital $q$, and those two qubits must be physically adjacent on the chip."

- On **Heron (`ibm_kingston`)**, the α and β qubit chains only touch at every 4th site, so we used `[(p, p) for p in range(0, num_orbitals, 4)]` truncated to 5 pairs (there are only 5 hardware-adjacent α–β crossings).
- On **Nighthawk (`ibm_miami`)**, the α and β rails can be laid next to each other so that $p$-th qubit on the α chain can interact with the $p$-th qubit on the β chain for *every* $p$. There is no need to skip indices.

**Hint:**
- **(Recommended)** You can make 24 qubit pairs (truncate at 24) rather than pairing all qubits. This makes it easier to map the circuit to the Nighthawk topology.
- You may notice that you cannot arrange every α and β pair side-by-side, but don't worry about that at this stage.

</div>

In [ ]:
# Select alpha-beta interaction pairs according to the Nighthawk device topology
# ---- TODO : Task 1a ----
alpha_beta_indices_nh = []
# ---- End of TODO : Task 1a ----

print(alpha_beta_indices_nh)

In [ ]:
grade_lab4c_ex1a(alpha_beta_indices_nh)

In [ ]:
ucj_op_nh = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_nh),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# create an empty quantum circuit
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_nh = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit_nh.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# apply the UCJ operator to the reference state
circuit_nh.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_nh), qubits)

# investigate the circuit for execution
inner_circuit_nh = ffsim.qiskit.PRE_INIT.run(circuit_nh)
for i in range(2, 0, -1):
    inner_circuit_nh.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_nh.decompose().draw("mpl", fold=-1))

# insert final measurement
circuit_nh.measure_all()

<a id="exercise_1b"></a>
<div class="alert alert-block alert-success">

<b>Exercise 1b: Define the qubit layout for the Nighthawk device</b>

**Your Goal:** Provide `spin_a_layout` and `spin_b_layout` — lists of exactly `num_orbitals` (= 26) physical qubit indices each — that correctly map the α and β spin-orbital chains onto Nighthawk's hardware.

**Background:** `initial_layout` instructs the transpiler to place logical qubit $p$ on physical qubit `initial_layout[p]`. A good layout must satisfy two requirements simultaneously:

1. **Connectivity within each spin sector:** the same-spin nearest-neighbor interactions `alpha_alpha_indices = [(p, p+1) ...]` require physical qubits $p$ and $p+1$ within each list to be hardware-adjacent (connected by a two-qubit gate line on the chip). Each list should therefore trace a *connected path* through the device graph.

2. **α–β adjacency:** according to the pairs provided in exercise 1a, place the pairs as close to each other as possible. It is perfectly fine to use a few swaps if needed.

**Hint:**
- Not every α and β pair will sit side-by-side. Applying a few swap operations is fine, as long as the swaps can operate in parallel, keeping the circuit depth minimal.
- **(Optional)** If you want to tackle a more advanced challenge, try pairing all 26 qubits in the previous exercise and mapping them to the Nighthawk. The passing criteria is quite challenging and the grader result may fluctuate with the transpiler seed. Try a few different seeds to get a better transpilation result.
</div>

In [ ]:
# select spin a and b layout according to the device topology
# ---- TODO : Task 1b ----
spin_a_layout = [

]
spin_b_layout = [
    
]
initial_layout = spin_a_layout + spin_b_layout
# ---- End of TODO : Task 1b ----

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_nh, initial_layout=initial_layout
)

# without PRE_INIT passes
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_nh.count_ops()}")

# with PRE_INIT passes
# We will use the circuit generated by this pass manager for hardware execution
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_nh.count_ops()}")

In [ ]:
grade_lab4c_ex1b(initial_layout, alpha_beta_indices_nh, seed=None)

## 3. Quantum Sampling

With the circuit mapped to Nighthawk's topology, we submit 2,000 shots to the device.
Each **shot** measures all `2*num_orbitals` qubits and returns one **bitstring** — a binary snapshot of which orbitals are occupied at that moment.
Physically, each bitstring is *one proposed electron configuration*: a candidate for a basis state of the ground state.

The collection of all 2,000 bitstrings, together with how often each distinct configuration appears, is the raw material that SQD will process.

> **Note on runtime:** Running a real hardware job takes 10 seconds on `ibm_miami` or 2 seconds on `ibm_kingston`.
> To keep this notebook self-contained, the cell below submits the job and saves the `job_id`.
> The *following* cell retrieves the pre-computed results from that saved ID — you do not need to resubmit.

In [ ]:
# # Option 1: if you have an access to the Nighthawk device, try running the square-lattice circuit.
# sampler = Sampler(mode=backend_nh)
# job = sampler.run([isa_circuit_nh], shots=2_000)

# # Option 2: if you don't have an access to the Nighthawk device, you can run Heron fitted circuit.
# sampler = Sampler(mode=backend_hr)
# job = sampler.run([isa_circuit_hr], shots=2_000)

job_id = job.job_id()
print(f"Submitted job {job_id}")

The cell below retrieves the completed job and converts the raw measurement outcomes into two arrays:

- `raw_bitstrings` — integer array of shape `(n_unique, 2*num_orbitals)` where each row is a distinct bitstring (each element is 0 or 1).
- `raw_probs` — float array of length `n_unique` giving the empirical probability of each bitstring; `raw_probs[k]` is the fraction of shots that produced `raw_bitstrings[k]`.

In [ ]:
# Retrieve job using the job id
job_id = "d8oks6e8aqlc73egmkl0"
job = service.job(job_id)
primitive_result = job.result()
pub_result = primitive_result[0]
bit_array = pub_result.data.meas

# Convert BitArray into bitstring and probability arrays
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)